# Week 3 Guided Exercise — Solution

This notebook matches the student exercise. Use it for teacher reference or release it after submission. Every executable line is explained in plain English.

## Part A — Load and inspect

In [ ]:
# Import pandas and use the standard short name pd.
import pandas as pd
# Import StringIO so text in this notebook can behave like a CSV file.
from io import StringIO
# Store a small, intentionally messy student dataset as CSV text.
csv_text = """student_id,name,grade,math_pre,math_post,science_pre,science_post,study_hours,club
S01,Ava,9,78,88,82,90,4.5,Robotics
S02,Ben,10,85,91,80,89,5.0,AI Club
S03,Chloe,9,92,95,94,96,6.0,Science
S04,Diego,11,68,79,72,81,3.5,robotics
S05,Emma,10,88,,86,92,5.5,Art
S06,Finn,9,74,83,70,,4.0,AI Club
S07,Grace,11,90,94,91,95,6.5,Science
S08,Hassan,10,65,76,69,78,3.0, Robotics 
S09,Ivy,9,81,87,84,90,4.5,Art
S10,Jayden,11,77,85,75,84,five,AI Club
S10,Jayden,11,77,85,75,84,five,AI Club
S11,Kai,10,83,90,88,93,5.0,science
"""
# Read the embedded CSV text into a Pandas DataFrame.
students = pd.read_csv(StringIO(csv_text))
# Display the DataFrame.
students

### Task 1 — Data audit

In [ ]:
# Print the number of rows and columns.
print("Shape:", students.shape)
# Display the column names.
print("Columns:", students.columns.tolist())
# Display the data type of each column.
display(students.dtypes)
# Count missing values in every column.
display(students.isna().sum())
# Count completely duplicated rows.
print("Duplicate rows:", students.duplicated().sum())

**Audit findings:** The raw table has 12 rows, including one duplicate. `math_post` and `science_post` each contain a missing value. `study_hours` is stored as text because one value is `five`. Club names contain inconsistent capitalization and spaces.

## Part B — Series, selection, and filtering

In [ ]:
# Select one column as a Series.
math_post_scores = students["math_post"]
# Print the Python type of the selection.
print(type(math_post_scores))
# Select four columns as a DataFrame.
math_view = students[["name", "grade", "math_pre", "math_post"]]
# Display the first five rows of the smaller DataFrame.
display(math_view.head())

### Task 2 — Answer targeted questions

In [ ]:
# Filter math pre-scores below 80 and select three columns by name.
display(students.loc[students["math_pre"] < 80, ["name", "grade", "math_pre"]])
# Filter grade 10 students whose science pre-score is at least 80.
display(students.loc[(students["grade"] == 10) & (students["science_pre"] >= 80), ["name", "grade", "science_pre"]])
# Select the first four rows and first five columns by position.
display(students.iloc[0:4, 0:5])

## Part C — Sort and clean

In [ ]:
# Sort science post-scores from highest to lowest.
science_ranking = students.sort_values(by="science_post", ascending=False)
# Display names and sorted science post-scores.
display(science_ranking[["name", "science_post"]])
# Copy the raw DataFrame before cleaning.
clean_students = students.copy()
# Remove extra spaces around club names.
clean_students["club"] = clean_students["club"].str.strip()
# Make club capitalization consistent.
clean_students["club"] = clean_students["club"].str.title()
# Convert study hours to numeric and replace invalid text with NaN.
clean_students["study_hours"] = pd.to_numeric(clean_students["study_hours"], errors="coerce")
# Remove completely duplicated rows.
clean_students = clean_students.drop_duplicates()
# Fill a missing math post-score with the column median.
clean_students["math_post"] = clean_students["math_post"].fillna(clean_students["math_post"].median())
# Fill a missing science post-score with the column median.
clean_students["science_post"] = clean_students["science_post"].fillna(clean_students["science_post"].median())
# Fill missing study hours with the column median.
clean_students["study_hours"] = clean_students["study_hours"].fillna(clean_students["study_hours"].median())
# Reset the row index and discard old labels.
clean_students = clean_students.reset_index(drop=True)
# Print the cleaned shape.
print("Clean shape:", clean_students.shape)
# Display the cleaned data types.
display(clean_students.dtypes)
# Display remaining missing-value counts.
display(clean_students.isna().sum())
# Display standardized club frequencies.
display(clean_students["club"].value_counts())

## Part D — Analyze improvement

In [ ]:
# Calculate each student's math improvement.
clean_students["math_improvement"] = clean_students["math_post"] - clean_students["math_pre"]
# Calculate each student's science improvement.
clean_students["science_improvement"] = clean_students["science_post"] - clean_students["science_pre"]
# Calculate the row average across both improvement columns.
clean_students["average_improvement"] = clean_students[["math_improvement", "science_improvement"]].mean(axis=1)
# Sort students from greatest to smallest average improvement.
improvement_ranking = clean_students.sort_values(by="average_improvement", ascending=False)
# Display the evidence behind the ranking.
display(improvement_ranking[["name", "math_improvement", "science_improvement", "average_improvement"]])
# Calculate the mean math post-score.
average_math_post = clean_students["math_post"].mean()
# Calculate the mean science post-score.
average_science_post = clean_students["science_post"].mean()
# Find the row label of the greatest average improvement.
best_index = clean_students["average_improvement"].idxmax()
# Select the name at the winning row label.
most_improved_name = clean_students.loc[best_index, "name"]
# Select the improvement value at the winning row label.
most_improved_value = clean_students.loc[best_index, "average_improvement"]

## Part E — Final report

In [ ]:
# Print the average math post-score rounded to one decimal place.
print(f"Average math post-score: {average_math_post:.1f}")
# Print the average science post-score rounded to one decimal place.
print(f"Average science post-score: {average_science_post:.1f}")
# Print the most improved student with one decimal place.
print(f"Most improved student: {most_improved_name} ({most_improved_value:.1f} points)")

## Sample conclusion

Science had the higher post-test average, about **88.4**, compared with math at about **86.6**. **Hassan** improved the most, gaining an average of **10.0 points** across math and science. The improvement table also shows that every student had a positive average change after missing values were filled. However, this dataset contains only 11 unique students, and median imputation changed some values. The analysis shows association and change, but it does not prove that study hours or any other factor caused the improvement.

## Reflection

Median imputation has strong potential to change the findings because it inserts estimated post-test values for students with missing results. It is simple and resists extreme values, but it can hide uncertainty and make the distribution look less variable than it really is.